[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KFUPM-JRCAI/star-instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [6]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/star-instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


In [8]:
TAWJEEH_DATASET_NAME = 'ArEntail'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArEntail_experimental'
MODEL_PATH = "/raid_storage/shared_models/Meta-Llama-3.1-8B"
TASK_NAME='NLI'

In [9]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [10]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14869,
  'tags': [],
  'name': 'answer keyword at the end of the prompt',
  'task': {'name': 'multiple choice'},
  'status': 'APPROVED',
  'template': 'This is a question. Select the correct answer!\r\n\r\nQuestion: \r\n{{Question}}\r\n\r\nChoices:\r\n{% set choices = [A,B,C,D] %}\r\n{% for choice in choices %}\r\n{% if choice and choice.strip %}\r\n{{ answer_choices[loop.index0] }}. {{choice}}\r\n{% endif %}\r\n{% endfor %}\r\nAnswer:\r\n|||\r\n{{answer_choices[answer_choices.index(answer)]}}',
  'dataset_name': 'arbml/ArabicMMLU',
  'dataset_subset': 'default',
  'answer_choices': ['A', 'B', 'C', 'D', 'E'],
  'text_direction': 'ltr'},
 {'id': 14865,
  'tags': [],
  'name': 'QA_stance_on_topic',
  'task': {'name': 'stance detection'},
  'status': 'SUBMITTED',
  'template': "Question: If someone says the following in Arabic {{text}} about {{target}}, do you think the person is 'against', 'favor' or 'neither'? Answer: \r\n|||\r\n{{answer_choices[stance]}}",
  'dataset_name': 'ar

In [11]:
len(prompts)

332

In [12]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

188

## Finetuning

### Get the dataset prompts

In [13]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

8

In [14]:
SELECTED_PROMPTS_IDS = [
    14581,   
    14816,
    14818,
    14819,
    14820,
]

In [15]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [16]:
import datasets

In [17]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 1000
    })
})

### Merge the prompts

In [18]:
from jinja2 import Environment, StrictUndefined

In [19]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [24]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [25]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "زعيم الحوثيين يعلن عن خمس جبهات لمواجهة عاصفة الحزم ," logically follows from the premise, "اليمن.. زعيم الحوثيين يدعو لرفد جبهات القتال بالمال والرجال والإرياني يؤكد فشل مفاوضات أممية مع الجماعة بشأن صافر." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
not entail


In [26]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

1000.0

In [27]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/5000 [00:00<?, ?it/s]

rending Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
|||
{{ answer_choices[label] }} sample index: 0
rending Task: Determine if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Carefully assess whether the premise provides enough support for the hypothesis. 

Answer with "entails" if the hypothesis logically follows, or "not entail" if it does not. Provide only your answer.
|||
{{ answer_choices[label] }} sample index: 1000
rending Task: Determine if the hypothesis follows logically from the premise. Follow these steps to reach a conclusion.

Steps:
1. Understand the

5000

## Finetune the LLM

In [28]:
GLOBAL_SEED = 42

In [29]:
import random
random.seed(GLOBAL_SEED)

In [30]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [31]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [32]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "_name_or_path": "/raid_storage/shared_models/Meta-Llama-3.1-8B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "voca

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing LlamaForCausalLM.

All the weights of LlamaForCausalLM were initialized from the model checkpoint at /raid_storage/shared_models/Meta-Llama-3.1-8B.
If your task is similar to the task the model of the checkpoint was trained on, you can already use LlamaForCausalLM for predictions without further training.
loading configuration file /raid_storage/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}

loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file /raid_storage/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config Ge

In [33]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    # each sample should be two lines only, the first line is for the inputs and the last is the output
    # we did the join for generalization
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    prefix += '\nThe answer is:'
    # prefix = re.sub(r'\s+', ' ', prefix).strip()
    # outputs are always the last line
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(4500,
 500,
 [('Based on the premise: منحه فرصة للترشح للرئاسة.. قاض يلغي إجراءات ملاحقة دا سيلفا في قضيتين, does the following hypothesis logically follow?\n\nHypothesis: ألغى قاض في المحكمة العليا البرازيلية إجراءات  ضد الرئيس السابق \n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' entails'),
  ('Based on the premise: زحف نحو الحدود ومظاهرات لا تتوقف.. الأردن يغضب للقدس وفلسطين ودعوات لقطع العلاقات مع إسرائيل, does the following hypothesis logically follow?\n\nHypothesis: احتجاجات لبنانية قرب الحدود تضامنا مع فلسطين\n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' not entail'),
  ('Welcome to the "Logic Match Game!"\n\nTask: In this game, your challenge is to decide if the statement, "الريال اليمني يتراجع إلى أدنى مستوى في تاريخه أمام الدولار," logically follows from the premise, "العملة اليمنية عند أدنى مستوى أمام الدولار." Think carefully: does the premise support the hypothesi

In [34]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'v_proj', 'q_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


loading configuration file /raid_storage/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 128256
}



{'eval_loss': 1.9076472520828247, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 13.7839, 'eval_samples_per_second': 36.274, 'eval_steps_per_second': 2.322}


***** Running training *****
  Num examples = 4,500
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 2,820
  Number of trainable parameters = 6,815,744


...

{'eval_loss': 0.17566744983196259, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 8.3208, 'eval_samples_per_second': 60.09, 'eval_steps_per_second': 3.846, 'epoch': 4.432624113475177}



***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


{'eval_loss': 0.19992105662822723, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 8.3238, 'eval_samples_per_second': 60.069, 'eval_steps_per_second': 3.844, 'epoch': 5.319148936170213}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.10562781989574432

In [35]:
exit()